In [1]:
import pandas as pd
from pathlib import Path
from bike_sharing.feature_engineering import (
    create_date_features,
    create_cyclical_encodings,
    engineer_lag_features,
)

PREPROCESSED_DF_PATH = Path("../data/processed/hour_preprocessed.parquet")
df = pd.read_parquet(PREPROCESSED_DF_PATH)

In [2]:
# look at the preprocessed DataFrame
df.head()

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16
1,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,40
2,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,32
3,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13
4,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1


In [3]:
# shape
df.shape

(17379, 14)

In [4]:
# info
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   dteday      17379 non-null  datetime64[us]
 1   season      17379 non-null  int64         
 2   yr          17379 non-null  int64         
 3   mnth        17379 non-null  int64         
 4   hr          17379 non-null  int64         
 5   holiday     17379 non-null  int64         
 6   weekday     17379 non-null  int64         
 7   workingday  17379 non-null  int64         
 8   weathersit  17379 non-null  int64         
 9   temp        17379 non-null  float64       
 10  atemp       17379 non-null  float64       
 11  hum         17379 non-null  float64       
 12  windspeed   17379 non-null  float64       
 13  cnt         17379 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(9)
memory usage: 1.9 MB


In [5]:
# check missing values
assert df.isna().sum().sum() == 0

## Create date-related features

In [6]:
df = create_date_features(df)

## Create cyclical encodings

In [7]:
df = create_cyclical_encodings(df)

## Engineer Lag Features

In [8]:
df = engineer_lag_features(df)

## Look at the final DataFrame

In [9]:
df.head()

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,...,cnt,year,day,hour_sin,hour_cos,month_sin,month_cos,cnt_lag_1,cnt_lag_24,cnt_lag_168
0,2011-01-01,1,0,1,0,0,6,0,1,0.24,...,16,2011,1,0.000000,1.000000,0.5,0.866025,NaN,NaN,NaN
1,2011-01-01,1,0,1,1,0,6,0,1,0.22,...,40,2011,1,0.258819,0.965926,0.5,0.866025,16.0,NaN,NaN
2,2011-01-01,1,0,1,2,0,6,0,1,0.22,...,32,2011,1,0.500000,0.866025,0.5,0.866025,40.0,NaN,NaN
3,2011-01-01,1,0,1,3,0,6,0,1,0.24,...,13,2011,1,0.707107,0.707107,0.5,0.866025,32.0,NaN,NaN
4,2011-01-01,1,0,1,4,0,6,0,1,0.24,...,1,2011,1,0.866025,0.500000,0.5,0.866025,13.0,NaN,NaN


In [10]:
# shape
df.shape

(17379, 23)

In [11]:
# check NaN values
df.isna().sum()

dteday           0
season           0
yr               0
mnth             0
hr               0
holiday          0
weekday          0
workingday       0
weathersit       0
temp             0
atemp            0
hum              0
windspeed        0
cnt              0
year             0
day              0
hour_sin         0
hour_cos         0
month_sin        0
month_cos        0
cnt_lag_1        1
cnt_lag_24      24
cnt_lag_168    168
dtype: int64

In [12]:
# drop NaN values which are just a few
df = df.dropna()

In [13]:
# check NaN values
df.isna().sum()

dteday         0
season         0
yr             0
mnth           0
hr             0
holiday        0
weekday        0
workingday     0
weathersit     0
temp           0
atemp          0
hum            0
windspeed      0
cnt            0
year           0
day            0
hour_sin       0
hour_cos       0
month_sin      0
month_cos      0
cnt_lag_1      0
cnt_lag_24     0
cnt_lag_168    0
dtype: int64

In [14]:
# move the target to the end
col_to_move = df.pop("cnt")
df.insert(len(df.columns), "cnt", col_to_move)

In [15]:
# save the feature engineered DataFrame
df.to_parquet("../data/processed/hour_feature_engineered.parquet", index=False)